In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# 전체 CSV 로딩
volume_path = "/Volumes/bronze_api/agrofood_shipmentsequel/volumn/*.csv"
df = spark.read.option("header", True).option("inferSchema", True).csv(volume_path)

print(f"행 개수: {df.count():,}")
print(f"컬럼 개수: {len(df.columns)}")
print(f"\n컬럼 목록:")
for c in df.columns:
    print(f"  - {c}")


# 기초통계 (수치형)
# display(df.select(price_cols).describe())


# # 가격 관련 컬럼 숫자형으로 변환
# price_cols = [c for c in df.columns if 'prc' in c]
# numeric_cols = ['ctgry_cd', 'grd_cd', 'item_cd', 'se_cd', 'unit_sz', 'vrty_cd'] + price_cols

# df2 = df
# for col_name in numeric_cols:
#     df2 = df2.withColumn(col_name, F.col(col_name).cast(DoubleType()))

In [0]:
# 지역별 품목별 도·소매 가격정보 기준 컬럼 한글명 매핑
col_kor_map = {
    "exmn_ymd": "조사일자",
    "se_cd": "구분코드",
    "se_nm": "구분명",
    "ctgry_cd": "부류코드",
    "ctgry_nm": "부류명",
    "item_cd": "품목코드",
    "item_nm": "품목명",
    "vrty_cd": "품종코드",
    "vrty_nm": "품종명",
    "grd_cd": "등급코드",
    "grd_nm": "등급명",
    "sgg_cd": "시군구코드",
    "sgg_nm": "시군구명",
    "unit": "단위",
    "unit_sz": "단위크기",
    "exmn_dd_min_prc": "조사일최저가격",
    "exmn_dd_cnvs_min_prc": "조사일환산최저가격",
    "exmn_dd_avg_prc": "조사일평균가격",
    "exmn_dd_cnvs_avg_prc": "조사일환산평균가격",
    "exmn_dd_max_prc": "조사일최고가격",
    "exmn_dd_cnvs_max_prc": "조사일환산최고가격"
}

total = df.count()

# 널값이 '', 'null', 'None' 등 문자열로 들어간 경우도 포함하여 체크
def is_effective_null(col):
    return (
        F.col(col).isNull() |
        (F.trim(F.col(col)) == "") |
        (F.lower(F.col(col)) == "null") |
        (F.lower(F.col(col)) == "none")
    )

notnull_counts = df.select(
    [
        F.sum(F.when(~is_effective_null(c), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]
).toPandas().T

notnull_counts.columns = ['notnull_count']
notnull_counts['notnull_pct'] = (notnull_counts['notnull_count'] / total * 100).round(2)
notnull_counts['null_count'] = total - notnull_counts['notnull_count']
notnull_counts['null_pct'] = (notnull_counts['null_count'] / total * 100).round(2)
notnull_counts['col_name'] = notnull_counts.index
notnull_counts['col_kor'] = notnull_counts['col_name'].map(col_kor_map).fillna("")

notnull_counts = notnull_counts[['col_name', 'col_kor', 'notnull_count', 'null_count', 'null_pct']]

display(notnull_counts)

In [0]:
from pyspark.sql import functions as F

cat_cols = [ "whsl_mrkt_cd", "corp_cd", "gds_lclsf_cd", "gds_mclsf_cd", "gds_sclsf_cd", "unit_cd", "unit_qty" ]

print("범주형 컬럼 고유값 수")
for c in cat_cols:
    cnt = df.select(c).distinct().count()
    print(f"  {c}: {cnt}개")

display(df.groupBy("whsl_mrkt_cd", "whsl_mrkt_nm").count().orderBy(F.desc("count")))   # 도매시장명
display(df.groupBy("corp_cd", "corp_nm").count().orderBy(F.desc("count")))               # 법인명
display(df.groupBy("gds_lclsf_cd", "gds_lclsf_nm").count().orderBy(F.desc("count")))     # 상품대분류명
display(df.groupBy("gds_mclsf_cd", "gds_mclsf_nm").count().orderBy(F.desc("count")))     # 상품중분류명
display(df.groupBy("gds_sclsf_cd", "gds_sclsf_nm").count().orderBy(F.desc("count")))     # 상품소분류명
display(df.groupBy("unit_cd", "unit_nm").count().orderBy(F.desc("count")))               # 단위명
display(df.groupBy("unit_qty").count().orderBy(F.desc("count")))                         # 단위물량

In [0]:
from pyspark.sql import functions as F

cat_cols = [ "unit_cd", "unit_nm"]

print("범주형 컬럼 고유값 수")
for c in cat_cols:
    cnt = df.select(c).distinct().count()
    print(f"  {c}: {cnt}개")

#display(df.groupBy("whsl_mrkt_cd", "whsl_mrkt_nm").count().orderBy(F.desc("count")))   # 도매시장명
#display(df.groupBy("corp_cd", "corp_nm").count().orderBy(F.desc("count")))               # 법인명
#display(df.groupBy("gds_lclsf_cd", "gds_lclsf_nm").count().orderBy(F.desc("count")))     # 상품대분류명
#display(df.groupBy("gds_mclsf_cd", "gds_mclsf_nm").count().orderBy(F.desc("count")))     # 상품중분류명
#display(df.groupBy("gds_sclsf_cd", "gds_sclsf_nm").count().orderBy(F.desc("count")))     # 상품소분류명
display(df.groupBy("unit_cd", "unit_nm").count().orderBy(F.desc("count")))               # 단위명
# display(df.groupBy("unit_qty").count().orderBy(F.desc("count")))                         # 단위물량